# Setup and Import

In [1]:
import os
import time
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
import tifffile as tiff
import gc
import rasterio
import tempfile
from osgeo import gdal
from scipy import ndimage
from scipy.ndimage import label, generic_filter
from rasterio.shutil import copy

# Globals

## Constants and paths

Configuration for this step (step 5: threshold the index raster into a vegetation
mask, then remove small "clumps" below `min_pxl_num` pixels): which municipality
(`procom`), index, flight year and thresholding `method` to use, the NoData
conventions on input (`-20`, from step 1) and output (`255`), and the minimum
clump size to keep (`min_pxl_num`, ~100 m²). Input/output paths are under
`output/<procom>/`. The threshold for `method` is read from the summary CSV
produced by steps 3/4 (KDE/KMeans/Australian).

In [2]:
start_t = time.time()

input_nodata_value = -20   # NoData value used by step 1 for the index rasters
mask_nodata_value = 255    # NoData value for the output classification mask
seq_size = 100000000
pxl_area = 0.04 # in m^2; pixel size = 20 cm = 0.2 m
save_output_masks = True
index = 'NDVI_red'  # NDVI_red; ENDVI; NDVI_blue
procom = "059033"
method = 'Australian' # ['KDE', 'KMeans', 'KMedians', 'Australian']
city_dict = {'039014': 'Ravenna', 
             '065116': 'Salerno', 
             '031007': 'Gorizia', 
             '038008': 'Ferrara', 
             '051002': 'Arezzo', 
             '048017': 'Firenze',
             '010025': 'Genova',
             '037006': 'Bologna',
             '032006': 'Trieste',
             '015146': 'Milano',
             '063049': 'Napoli',
             '082053': 'Palermo',
             '001272': 'Torino',
             '092009': 'Cagliari',
             '058091': 'Roma',
             '027042': 'Venezia',
             '080063': 'Reggio Calabria',
             '087015': 'Catania',
             '083048': 'Messina',
             '059033': 'Ventotene'}
city_name = city_dict[procom]
flight_year = '2022'
min_pxl_num = 2500 # equivalent to 100m^2
if (procom=='080063') & (flight_year == '2012'): #RC 2012 has different orthoimages: 50cm/pxl
   print('RC 2012; 50cm/pxl')
   pxl_area=0.25
   min_pxl_num = 400 # equivalent to 100m^2

# Input file produced by step 1 (crop + evaluate indices + mosaic), e.g.:
# C:\Users\UTENTE\Downloads\JOS_areeverdi\output\059033\Ventotene_d1-059033-NDVI_red-2022.tif
# This is the city-wide mosaic (step 1 deletes the intermediate per-tile files
# once the mosaic has been written), not a per-tile raster.
indices_file_name = city_name + '-' + procom + '-' + index + '-' + flight_year + '.tif'

# Summary stats CSV produced by steps 3/4 (KDE/KMeans/Australian), holding the
# threshold this step reads for `method`.
stat_file_name = city_name + '-' + procom + '-' + index + '-' + flight_year + '.csv'
clump_file_name = city_name + '-' + procom + '-' + index + '-' + flight_year + '-' + method + '_clump.tif'
mask_file_name = city_name + '-' + procom + '-' + index + '-' + flight_year + '-' + method + '_mask.tif'
show_images = False # Ferrara OK; Firenze KO

# All inputs/outputs for this procom live under output/<procom>/
base_path = "C:/Users/UTENTE/Downloads/JOS_areeverdi/"
output_path = base_path + "output/" + procom + "/"

print('output_path: ', output_path)

output_path:  C:/Users/UTENTE/Downloads/JOS_areeverdi/output/059033/


In [3]:
print(os.path.join(output_path, indices_file_name))
print(os.path.join(output_path, clump_file_name))
print(os.path.join(output_path, mask_file_name))
print(os.path.join(output_path, stat_file_name))

C:/Users/UTENTE/Downloads/JOS_areeverdi/output/059033/Ventotene_d1-059033-NDVI_red-2022.tif
C:/Users/UTENTE/Downloads/JOS_areeverdi/output/059033/Ventotene_d1-059033-NDVI_red-2022-Australian_clump.tif
C:/Users/UTENTE/Downloads/JOS_areeverdi/output/059033/Ventotene_d1-059033-NDVI_red-2022-Australian_mask.tif
C:/Users/UTENTE/Downloads/JOS_areeverdi/output/059033/Ventotene_d1-059033-NDVI_red-2022.csv


In [4]:
print(index)
print(procom)
print(city_name)
print(flight_year)
print(method)
print(indices_file_name)

NDVI_red
059033
Ventotene_d1
2022
Australian
Ventotene_d1-059033-NDVI_red-2022.tif


## Functions

I/O and processing helpers used throughout the notebook:

- `read_geotiff` / `read_geotiff_as_ndarray`: open a GeoTIFF with GDAL, optionally as a plain numpy array.
- `label_2d_binary_mask`: label connected clumps of vegetation pixels (4-neighborhood connectivity) via `scipy.ndimage.label`.
- `clump_to_mask`: keep only clumps larger than `min_pxl_num` pixels (i.e. areas > 100 m²), discarding smaller ones.
- `memory_map`: fallback loader for very large, compressed rasters that don't fit comfortably in RAM.
- `load_mosaic`: generic loader that dispatches on file extension (`.tif`, `.npy`, `.npz`).
- `write_compressed_mask`: legacy writer (fixed NoData = -9999), kept for reference but not used below.
- `write_compressed_mask_with_nodata`: the writer actually used to export the final clumped mask — writes a compressed, tiled, Byte GeoTIFF block by block, declaring the given NoData value on the output band.

In [5]:

def read_geotiff(filename):
    ds = gdal.Open(filename)
    return ds

def read_geotiff_as_ndarray(filename):
    ds = gdal.Open(filename)
    band = ds.GetRasterBand(1)
    arr = band.ReadAsArray()
    return arr

def label_2d_binary_mask(image_masked):
    # Pixel connectivity: 4-neighborhood ("cross" pattern) -- two vegetation
    # pixels are considered part of the same clump only if they touch on an
    # edge (not diagonally).
    connectivity_array = np.array([ [0,1,0],
                                    [1,1,1],
                                    [0,1,0]  ], dtype=np.int8)
    
    # label() assigns a unique id to every group of connected pixels ("clumps")
    # according to connectivity_array.
    labeled_array, num_features = label(image_masked, structure= connectivity_array)
    # print(labeled_array, '\n')
    return labeled_array, num_features

def clump_to_mask(clump_array):
  # see: https://stackoverflow.com/questions/65405390/how-to-use-np-unique-on-big-arrays

  # First: for each labeled clump, count how many pixels it has, and keep only
  # the pixels belonging to clumps larger than min_pxl_num (i.e. areas > 100 m^2)
  clump_bin_array = np.zeros(clump_array.shape, dtype=np.int8)
  for i, loc in enumerate(ndimage.find_objects(clump_array)):
      loc_values, loc_counts = np.unique(clump_array[loc], return_counts=True) # original from web
      for idx in range(loc_values.shape[0]):
          
          # Second: keep this clump only if it is large enough
          if loc_counts[idx]>min_pxl_num:
              clump_bin_array[loc] = np.logical_or(clump_bin_array[loc], np.where(clump_array[loc]==loc_values[idx], 1, 0))

  #print(clump_bin_array)
  #print(clump_bin_array.shape)
  #print(clump_bin_array.dtype)
  return clump_bin_array

def memory_map(src_path):
  dst_path = src_path.replace(".tif", "_uncompressed.tif")

  with tempfile.NamedTemporaryFile(suffix=".tif", delete=True) as tmp:
    with rasterio.open(src_path) as src:
        profile = src.profile
        profile.update(
            compress='none',
            tiled=False
        )
        copy(src, dst_path, **profile)   

  return tiff.memmap(dst_path)

def load_mosaic(path_to_mosaic, mosaic_file_name):
    ext = os.path.splitext(mosaic_file_name)[1][1:]
    print(ext)
    if ext == 'tif':
        print('Loading GeoTiff file')
        try:
            mosaic_array = read_geotiff_as_ndarray(os.path.join(path_to_mosaic, mosaic_file_name))
        except:
            mosaic_array = memory_map(os.path.join(path_to_mosaic, mosaic_file_name))
            
    elif ext == 'npy':
        print('Loading ndarray')
        mosaic_array = np.load(os.path.join(path_to_mosaic, mosaic_file_name))
    elif ext == 'npz':
        ds = np.load(os.path.join(path_to_mosaic, mosaic_file_name))
        mosaic_array = ds["array"]
        del ds
        gc.collect()

    return mosaic_array, ext

"""
def show_variables():
    # Debugging helper: lists local and global variables sorted by in-memory size
    # (kept commented out, as in the original notebook)
    variables_dict = {**globals(), **locals()}
    variables=[]
    for name, value in variables_dict.items():
        variables.append([name, sys.getsizeof(value)])
    print(pd.DataFrame(variables).sort_values(1,ascending=False))
"""

def write_compressed_mask(mask_file_name, mask_array, ds_mask):
    rows, cols = mask_array.shape
    driver = gdal.GetDriverByName("GTiff")
    ds = driver.Create(
        mask_file_name,
        cols,
        rows,
        1,
        gdal.GDT_Byte,
        options=[
            "COMPRESS=ZSTD",
            "ZSTD_LEVEL=9",
            "PREDICTOR=2",
            "TILED=YES",
            "BIGTIFF=YES",
            "BLOCKXSIZE=512",
            "BLOCKYSIZE=512",
            "NUM_THREADS=ALL_CPUS"
        ]
    )
    ds.SetProjection(ds_mask.GetProjection())
    ds.SetGeoTransform(ds_mask.GetGeoTransform())
    band = ds.GetRasterBand(1)
    band.SetNoDataValue(-9999)
    band.WriteArray(mask_array)
    band.FlushCache()
    ds = None
    print("GeoTIFF saved.")

def write_compressed_mask_with_nodata(mask_file_name, mask_array, ds_mask, mask_nodata_value=255):
    """
    Writes mask_array (already containing mask_nodata_value wherever the source
    pixel was NoData) as a compressed Byte GeoTIFF, reusing projection/geotransform
    from ds_mask (the source index raster). NoData handling is done upstream, on
    the numpy array itself, so this function just needs to declare the NoData
    value on the output band and write blockwise.
    """
    rows, cols = mask_array.shape

    driver = gdal.GetDriverByName("GTiff")
    ds = driver.Create(
        mask_file_name, 
        cols, 
        rows, 
        1, 
        gdal.GDT_Byte, 
        options=[
            "COMPRESS=ZSTD",
            "ZSTD_LEVEL=9",
            "PREDICTOR=2",
            "TILED=YES",
            "BIGTIFF=YES",
            "BLOCKXSIZE=512",
            "BLOCKYSIZE=512",
            "NUM_THREADS=ALL_CPUS"
        ]
    )
    
    # Copy georeferencing (projection + geotransform) from the source index raster
    ds.SetProjection(ds_mask.GetProjection())
    ds.SetGeoTransform(ds_mask.GetGeoTransform())
    
    band = ds.GetRasterBand(1)
    band.SetNoDataValue(mask_nodata_value)
    
    # Write block by block (512x512 pixels at a time) to keep peak memory low
    block_size = 512
    mask_array_u8 = mask_array.astype(np.uint8)

    for y in range(0, rows, block_size):
        y_size = min(block_size, rows - y)
        for x in range(0, cols, block_size):
            x_size = min(block_size, cols - x)
            out_block = mask_array_u8[y:y+y_size, x:x+x_size]
            band.WriteArray(out_block, x, y)

    # Close and flush to disk
    band.FlushCache()
    ds = None
    print("GeoTIFF successfully saved block by block: ", mask_file_name)

# Data
Load the summary stats CSV produced by steps 3/4 (KDE/KMeans/Australian), read the
threshold for the chosen `method`, load the index raster, and evaluate the
vegetation/non-vegetation mask.

### Load threshold from the step-3/4 CSV report

Reads the summary CSV produced by the previous steps (KDE, KMeans, Australian)
and picks the row matching the configured `method`, to reuse its threshold
and total pixel count rather than recomputing them here.

In [6]:
stat_file_df = pd.read_csv(os.path.join(output_path, stat_file_name),delimiter=";", index_col=0, decimal=',')
display(stat_file_df)

,city,method,fl. year,index,thres.,tot pix.,green pix.,green %,green m^2,green ha,elab. time
0,059033 Ventotene_d1,KDE,2022,NDVI_red,0.142857,9647428,4556477,47.229966,182259.08,18.225908,0.727198
1,059033 Ventotene_d1,KMeans (2 clusters),2022,NDVI_red,0.164751,9647428,4209685,43.635309,168387.40,16.838740,0.678105
2,059033 Ventotene_d1,Australian (real edg. = True; dil. = 5),2022,NDVI_red,0.226083,9647428,3189243,33.057961,127569.72,12.756972,26.606700


In [7]:
# Match the configured `method` against the first word of each row's method
# label (e.g. "KMeans (2 clusters)" -> "KMeans"), then read its threshold and
# total pixel count.
meths = stat_file_df['method'].values.tolist()
search_keys = [m.split()[0] for m in meths]
stat_file_df['search_keys'] = search_keys
print(search_keys)
print(method)

threshold = float(stat_file_df.loc[stat_file_df['search_keys']==method, 'thres.'].values[0])
tot_pxl = int(stat_file_df.loc[stat_file_df['search_keys']==method, 'tot pix.'].values[0])
print(threshold)
print(tot_pxl)

['KDE', 'KMeans', 'Australian']
Australian
0.2260826379060745
9647428


#### Load indices file

Loads the city-wide mosaic produced by step 1 (`indices_file_name`) into memory
as a numpy array via `load_mosaic`.

In [8]:
indices_values_array, ext = load_mosaic(output_path, indices_file_name)
print(indices_file_name)
print(indices_values_array.shape)
print(indices_values_array.dtype)

tif
Loading GeoTiff file


C:\Users\UTENTE\anaconda3\envs\geo\Lib\site-packages\osgeo\gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Ventotene_d1-059033-NDVI_red-2022.tif
(6526, 5309)
float32


In [9]:
x, y = indices_values_array.shape
indices_flatten = np.reshape(indices_values_array, (x*y))
tot_seq = indices_flatten.shape[0]
print(indices_flatten.shape)
print(indices_flatten.dtype)

# Track NoData locations *before* any binary classification/clumping is done.
# The classification and clumping pipeline below needs indices_flatten to stay
# a clean binary 0/1 array (scipy.ndimage.label and the logical_and/logical_or
# overlap-handling logic all assume strictly binary input), so NoData (-20)
# cannot be carried through as a third value. Instead we remember here which
# pixels were originally NoData, and re-apply mask_nodata_value (255) only at
# the very end, right before export.
nodata_flat_mask = (indices_flatten == input_nodata_value)

del indices_values_array
gc.collect()

(34646534,)
float32


0

#### Evaluate mask

Classifies every pixel as vegetation (`1`) or non-vegetation (`0`) using the
threshold loaded above. NoData locations were already recorded in
`nodata_flat_mask` (see the previous cell), so they can be safely restored at
export time without breaking the strictly-binary array the clumping pipeline
below relies on.

Note: for GE (Genova) about 5 mins.

In [10]:
k=1

# Classify each pixel as vegetation (1) or non-vegetation (0) using the threshold
# loaded above. NoData pixels are classified too (they will compare False against
# the threshold and end up as 0), but their original location was already saved
# in nodata_flat_mask, so they can be correctly restored to mask_nodata_value at
# export time without disturbing the binary 0/1 assumption used by the clumping
# pipeline below.
for i in range(0, tot_seq, seq_size):
    print('tile ' + str(k) + ': ' + str(i))
    if np.any(indices_flatten[i:i+seq_size]):
        indices_flatten[i:i+seq_size] = np.where((indices_flatten[i:i+seq_size]>=threshold) & (indices_flatten[i:i+seq_size] <= 1.0),1,0).astype(np.int8)
        print(np.max(indices_flatten[i:i+seq_size]))
        print(np.min(indices_flatten[i:i+seq_size]))
        print()
    else:
        print('Empty tile')
        print(indices_flatten[i:i+seq_size].shape)
    k+=1

indices_flatten = np.reshape(indices_flatten,(x,y,-1))
indices_flatten = np.squeeze(indices_flatten, axis=2)
nodata_flat_mask = np.reshape(nodata_flat_mask,(x,y,-1))
nodata_flat_mask = np.squeeze(nodata_flat_mask, axis=2)
print(indices_flatten.shape)
print(indices_flatten.dtype)

tile 1: 0
1.0
0.0

(6526, 5309)
float32


# Clumping by labeling
- first, split the thresholded mask into horizontal strips and label connected clumps in each
- second, evaluate the unique clump ids in the labeled strip
- third, keep only clumps with more than `min_pxl_num` pixels (i.e. areas greater than 100 m^2)
- fourth, repeat the first three steps for overlapping strips and take the OR of the overlapping region, so that a clump spanning a strip boundary isn't incorrectly cut in two
- fifth, to remove the non-green areas, build the final mask as the logical AND of the initial mask (thresholded index) and the large-clumps mask

In [11]:
# Preprocessing: remove isolated pixels from the image
# Note: for GE (Genova) nearly 25 hours would be needed; unfeasible.
#mask_array = remove_isolated_pixels(mask_array)

#### Note: for Roma about 91 min.; for GE (Genova) about 51 mins.; for RA (Ravenna) about 2 mins.

In [12]:
tile_h = 2000 # height of each horizontal strip, in pixels
n_pxl = 0
# First
for i in range(0, indices_flatten.shape[0], tile_h):
    print(i)
    clump_array, num_features = label_2d_binary_mask(indices_flatten[i:i+tile_h,:].astype(np.int8))
    print(clump_array.shape)
    print(clump_array.dtype)
    print(num_features)
    clump_bin = clump_to_mask(clump_array) # Second and Third
    print(clump_bin.shape)
    print(clump_bin.dtype)
    del clump_array
    del num_features
    gc.collect()

    # Fourth
    if (i>0) & (i<indices_flatten.shape[0]):
        print('here clumps on overlapping tile')
        row_min = int(i-(tile_h/2))
        row_max = int(min(i+(tile_h/2), indices_flatten.shape[0]))
        print(row_min)
        print(row_max)
        overlap_clump_array, overlap_num_features = label_2d_binary_mask(overlap_original_mask)
        overlap_clump_bin = clump_to_mask(overlap_clump_array)
        del overlap_clump_array
        del overlap_num_features
        gc.collect()
        indices_flatten[row_min:i,:] = np.logical_or(indices_flatten[row_min:i,:], overlap_clump_bin[:int(tile_h/2),:])
        indices_flatten[i:row_max,:] = np.logical_or(clump_bin[:int(tile_h/2),:], overlap_clump_bin[int(tile_h/2):tile_h,:])
        #Fifth (2)
        indices_flatten[row_min:row_max,:] = np.logical_and(indices_flatten[row_min:row_max,:], overlap_original_mask)
        n_pxl += np.sum(indices_flatten[row_min:row_max,:], dtype=np.int64)
        del overlap_original_mask
        del overlap_clump_bin
        gc.collect()
    overlap_original_mask = indices_flatten[int(i+(tile_h/2)):int(i+(3*tile_h/2)),:].astype(np.int8)
    if (i==0):
        #Fifth (1)
        indices_flatten[:int(tile_h/2),:] = np.logical_and(indices_flatten[:int(tile_h/2),:], clump_bin[:int(tile_h/2),:])
        n_pxl += np.sum(indices_flatten[:int(tile_h/2),:], dtype=np.int64)
        indices_flatten[int(tile_h/2):int(tile_h),:] = clump_bin[int(tile_h/2):int(tile_h),:]
    elif (i+tile_h>indices_flatten.shape[0]):
        indices_flatten[i+int(tile_h/2):,:] = np.logical_and(indices_flatten[i+int(tile_h/2):,:], clump_bin[int(tile_h/2):,:])
        n_pxl += np.sum(indices_flatten[i+int(tile_h/2):,:], dtype=np.int64)
    else:
        indices_flatten[int(i+tile_h/2):i+tile_h,:] = clump_bin[int(tile_h/2):int(tile_h),:]
    del clump_bin
    gc.collect()
    print(n_pxl)
    #show_variables()
    print()

print(indices_flatten)
print(indices_flatten.shape)
print(indices_flatten.dtype)


0
(2000, 5309)
int32
822
(2000, 5309)
int8
0

2000
(2000, 5309)
int32
6489
(2000, 5309)
int8
here clumps on overlapping tile
1000
3000
338536

4000
(2000, 5309)
int32
3351
(2000, 5309)
int8
here clumps on overlapping tile
3000
5000
2026239

6000
(526, 5309)
int32
535
(526, 5309)
int8
here clumps on overlapping tile
5000
6526
2765771

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
(6526, 5309)
float32


# Stats

Builds a one-row summary (method = `"<method> + clump"`) with the total valid
pixels, classified vegetation pixels/percentage/area (m² and ha) after
clump-size filtering, and elapsed time for this step. This row (`data_fr`) is
appended to the existing summary CSV in the Export section below.

In [13]:
end_t = time.time()
e_t = end_t - start_t

methods = [str(method) + ' + clump']
procom_city = procom + ' ' + city_name
perc = n_pxl/tot_pxl*100
m_2 = n_pxl * pxl_area
km_2 = m_2 * 1e-6
ha = m_2 * 1e-4

data_dict = {'city': procom_city, 
             'method': methods, 
             'fl. year': flight_year, 
             'index': index, 
             'tot pix.': [tot_pxl], 
             'green pix.': [n_pxl], 
             'green %': [perc], 
             'green m^2': [m_2], 
             'green ha': [ha],
             'elab. time': [e_t]}
data_fr = pd.DataFrame(data_dict)
display(data_fr)

,city,method,fl. year,index,tot pix.,green pix.,green %,green m^2,green ha,elab. time
0,059033 Ventotene_d1,Australian + clump,2022,NDVI_red,9647428,2765771,28.66848,110630.84,11.063084,12.842178


In [14]:
print('Total time: ', e_t)

Total time:  12.842177629470825


# Export

Writes the outputs of this step, all into `output_path` (`output/<procom>/`):

- `<city_name>-<procom>-<index>-<flight_year>-<method>_clump.tif` — final vegetation mask after clump-size filtering (only if `save_output_masks` is `True`). Byte raster: `0` = non-vegetation, `1` = vegetation, `255` = NoData, reusing the projection/geotransform of the source index raster.
- `<city_name>-<procom>-<index>-<flight_year>.csv` — summary stats (always written): this method's row is appended to the existing CSV (KDE/KMeans/Australian) if found, otherwise a new CSV is created with just this row.

In [15]:
if save_output_masks:
    ds_1 = read_geotiff(os.path.join(output_path, indices_file_name))
    print(ds_1.GetProjection())
    print(ds_1.GetGeoTransform())
    print(indices_flatten.shape)

PROJCS["RDN2008 / UTM zone 33N",GEOGCS["RDN2008",DATUM["Rete_Dinamica_Nazionale_2008",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1132"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","6706"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",15],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","7792"]]
(366955.1456853751, 0.2, 0.0, 4518024.42108661, 0.0, -0.2)
(6526, 5309)


In [16]:
if save_output_masks:
    # Restore NoData (255) at the pixels that were originally NoData in the input
    # index raster. This is done as the very last step, after classification and
    # clumping, so that the binary 0/1 assumptions used throughout that pipeline
    # were never broken by a third value.
    final_mask = indices_flatten.astype(np.uint8)
    final_mask[nodata_flat_mask] = mask_nodata_value

    write_compressed_mask_with_nodata(
        os.path.join(output_path, clump_file_name),
        final_mask,
        ds_1,
        mask_nodata_value
    )

    # Append this run's stats as a new row in the shared summary CSV, keeping
    # any rows already written by earlier steps (KDE/KMeans/Australian/...).
    csv_path = os.path.join(output_path, stat_file_name)
    if os.path.exists(csv_path):
        existing_df = pd.read_csv(csv_path, delimiter=';', index_col=0, decimal=',')
        existing_df = existing_df.drop(columns=['search_keys'], errors='ignore')
        combined_df = pd.concat([existing_df, data_fr], ignore_index=True)
    else:
        combined_df = data_fr
    combined_df.to_csv(csv_path, sep=';', decimal=',', index=True)
    print('Stats appended to: ', csv_path)

GeoTIFF successfully saved block by block:  C:/Users/UTENTE/Downloads/JOS_areeverdi/output/059033/Ventotene_d1-059033-NDVI_red-2022-Australian_clump.tif
Stats appended to:  C:/Users/UTENTE/Downloads/JOS_areeverdi/output/059033/Ventotene_d1-059033-NDVI_red-2022.csv
